# 01 - Baselines: Naive, Cumulative, Joint Training

These three are not "real" CL algorithms -- they're the reference points
every other strategy should be compared against:

- **Naive**: plain sequential fine-tuning, no CL mechanism at all -> the
  *lower bound* (expect heavy forgetting).
- **Cumulative**: retrains on all data seen so far at every experience ->
  strong but memory/compute-expensive.
- **Joint Training**: trains offline on the whole stream at once -> the
  *upper bound* / "if you didn't have a CL problem at all" reference.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import warnings; warnings.filterwarnings("ignore")

import torch
import pandas as pd
from avalanche.models import SimpleMLP
from avalanche.training import Naive, Cumulative, JointTraining
from avalanche.training.plugins import EvaluationPlugin
from avalanche.evaluation.metrics import accuracy_metrics, forgetting_metrics, loss_metrics

from bench_utils import make_synthetic_benchmark
from run_utils import run_strategy, _get_metric

BENCHMARK_CONFIG = dict(
    n_classes=10, n_experiences=5, feature_dim=64,
    n_per_class=250, class_sep=1.6, noise=1.0, seed=0,
)
benchmark = make_synthetic_benchmark(**BENCHMARK_CONFIG)

def new_model():
    return SimpleMLP(num_classes=benchmark.n_classes, input_size=benchmark.feature_dim,
                      hidden_size=64, hidden_layers=1, drop_rate=0.0)

def new_evaluator():
    return EvaluationPlugin(
        accuracy_metrics(experience=True, stream=True),
        forgetting_metrics(experience=True, stream=True),
        loss_metrics(stream=True),
        loggers=[],  # keep notebooks quiet; we read metrics from the returned dict instead
    )

all_rows = []


In [2]:
# --- Naive ---
model = new_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
strategy = Naive(model=model, optimizer=opt, criterion=torch.nn.CrossEntropyLoss(),
                  train_mb_size=32, train_epochs=3, eval_mb_size=128,
                  evaluator=new_evaluator(), device="cpu")
rows, final = run_strategy(strategy, benchmark, "Naive", "baseline")
all_rows += rows
print(final)


{'strategy': 'Naive', 'category': 'baseline', 'after_experience': 4, 'stream_acc': 0.306, 'stream_forgetting': 0.8574058919803601, 'train_seconds': 0.296475887298584}


In [3]:
# --- Cumulative ---
model = new_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
strategy = Cumulative(model=model, optimizer=opt, criterion=torch.nn.CrossEntropyLoss(),
                       train_mb_size=32, train_epochs=3, eval_mb_size=128,
                       evaluator=new_evaluator(), device="cpu")
rows, final = run_strategy(strategy, benchmark, "Cumulative", "baseline")
all_rows += rows
print(final)


{'strategy': 'Cumulative', 'category': 'baseline', 'after_experience': 4, 'stream_acc': 1.0, 'stream_forgetting': 0.0, 'train_seconds': 0.7165482044219971}


In [4]:
# --- Joint Training ---
# NOTE: JointTraining's .train() call trains on the *whole* stream at once
# (it doesn't map 1:1 onto "one experience at a time"), so we call it once
# rather than looping experience-by-experience like the other strategies.
model = new_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
strategy = JointTraining(model=model, optimizer=opt, criterion=torch.nn.CrossEntropyLoss(),
                          train_mb_size=32, train_epochs=3, eval_mb_size=128,
                          evaluator=new_evaluator(), device="cpu")
strategy.train(benchmark.train_stream)
res = strategy.eval(benchmark.test_stream)
all_rows.append({
    "strategy": "JointTraining", "category": "baseline",
    "after_experience": len(benchmark.train_stream) - 1,
    "stream_acc": _get_metric(res, "Top1_Acc_Stream"),
    "stream_forgetting": _get_metric(res, "StreamForgetting"),
})
print(all_rows[-1])


{'strategy': 'JointTraining', 'category': 'baseline', 'after_experience': 4, 'stream_acc': 1.0, 'stream_forgetting': 0.0}


In [5]:
df = pd.DataFrame(all_rows)
df.to_csv("../results/01_baselines.csv", index=False)
df


,strategy,category,after_experience,stream_acc,stream_forgetting
0,Naive,baseline,0,0.218,0.000000
1,Naive,baseline,1,0.302,0.477064
2,Naive,baseline,2,0.396,0.500000
3,Naive,baseline,3,0.254,0.894231
4,Naive,baseline,4,0.306,0.857406
5,Cumulative,baseline,0,0.218,0.000000
6,Cumulative,baseline,1,0.406,0.000000
7,Cumulative,baseline,2,0.614,0.000000
8,Cumulative,baseline,3,0.802,0.000000
9,Cumulative,baseline,4,1.000,0.000000
